# 02. 정제: 사고 원본 → 분석 가능한 형태

입력: `data/1_원본/accident.csv`, `data/1_원본/tunnel.csv` (01 노트북 결과)
출력: `data/2_중간/accident_clean.csv`, `data/2_중간/tunnel_expressway.csv`

하는 일
- 사고일자 + 사고시간 → `사고일시` (datetime). 시 단위인지 분 단위인지 자동 판별
- X/Y좌표 중 위도·경도 자동 판별 → `lat`, `lon`
- 숫자형 변환, 좌표 이상치 제거
- 터널은 고속도로만 남기고 입구·출구 좌표 정리

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW, INTERIM = ROOT / "data" / "1_원본", ROOT / "data" / "2_중간"
INTERIM.mkdir(parents=True, exist_ok=True)

acc = pd.read_csv(RAW / "accident.csv", dtype=str)
tun = pd.read_csv(RAW / "tunnel.csv", dtype=str)
print(acc.shape, tun.shape)

## 1. 사고일시 만들기

In [ ]:
def parse_date(s):
    s = s.str.strip().str.replace(r"[^0-9]", "", regex=True)   # 2023-01-01, 2023.01.01, 20230101 모두 → 20230101
    return pd.to_datetime(s, format="%Y%m%d", errors="coerce")

def parse_time(s):
    """'14', '14:30', '1430', '14시' 등을 (시, 분)으로. 분 정보가 없으면 분=0"""
    s = s.str.strip()
    digits = s.str.replace(r"[^0-9]", "", regex=True)
    has_min = digits.str.len() >= 3
    hour = np.where(has_min, digits.str[:-2], digits).astype(float)
    minute = np.where(has_min, digits.str[-2:], "0").astype(float)
    return pd.Series(hour, index=s.index), pd.Series(minute, index=s.index), has_min

acc["사고일"] = parse_date(acc["사고일자"])
hour, minute, has_min = parse_time(acc["사고시간"])
acc["has_min"] = has_min          # 분 정보가 있는 행인지 (05에서 태양 위치 계산 시각에 사용)
print(f"분 단위 정보가 있는 비율: {has_min.mean():.1%}")
acc["hour"] = hour
acc["사고일시"] = acc["사고일"] + pd.to_timedelta(hour, unit="h") + pd.to_timedelta(minute, unit="m")

bad = acc["사고일시"].isna() | (acc["hour"] > 23)
print("일시 변환 실패:", bad.sum())
acc.loc[bad, ["사고일자", "사고시간"]].head()

## 2. 위도·경도 판별 (한국: 위도 33~39, 경도 124~132)

In [ ]:
x = pd.to_numeric(acc["X좌표"], errors="coerce")
y = pd.to_numeric(acc["Y좌표"], errors="coerce")
if x.median() < 90:          # X가 30대면 X=위도
    acc["lat"], acc["lon"] = x, y
else:
    acc["lat"], acc["lon"] = y, x
print("X좌표 =", "위도" if x.median() < 90 else "경도")

ok = acc["lat"].between(33, 39) & acc["lon"].between(124, 132)
print("좌표 이상/결측:", (~ok).sum())

In [ ]:
for c in ["건수", "사망", "부상"]:
    acc[c] = pd.to_numeric(acc[c], errors="coerce").fillna(0).astype(int)
acc["이정_km"] = pd.to_numeric(acc["사고발생이정"].str.replace(r"[^0-9.]", "", regex=True), errors="coerce")

clean = acc[~bad & ok].copy()
clean.insert(0, "accident_id", range(len(clean)))
clean = clean[["accident_id", "사고일시", "has_min", "hour", "사고노선", "이정_km", "방향", "lat", "lon",
               "건수", "사망", "부상", "사고원인", "날씨"]]
clean.to_csv(INTERIM / "accident_clean.csv", index=False, encoding="utf-8-sig")
print(len(acc), "→", len(clean), "행 저장")
clean.head()

## 3. 터널: 고속도로만, 입구·출구 좌표

In [ ]:
print(tun["도로종류"].value_counts())

In [ ]:
t = tun[tun["도로종류"].str.contains("고속", na=False)].copy()
for c in ["터널시작점위도", "터널시작점경도", "터널종료점위도", "터널종료점경도", "터널연장"]:
    t[c] = pd.to_numeric(t[c], errors="coerce")
t = t.dropna(subset=["터널시작점위도", "터널시작점경도", "터널종료점위도", "터널종료점경도"])
t.insert(0, "tunnel_id", range(len(t)))
t = t.rename(columns={"터널시작점위도": "start_lat", "터널시작점경도": "start_lon",
                      "터널종료점위도": "end_lat", "터널종료점경도": "end_lon", "터널연장": "length_m"})
t = t[["tunnel_id", "터널명", "도로노선명", "도로노선번호", "도로노선방향", "start_lat", "start_lon",
       "end_lat", "end_lon", "length_m", "터널준공연도", "관리기관명"]]
t.to_csv(INTERIM / "tunnel_expressway.csv", index=False, encoding="utf-8-sig")
print(len(t), "개 고속도로 터널 저장")
t.head()